In [ ]:
import torch
from torch import nn

import torch_pruning as tp

import math

from snp import pruning, gram_matrices, importances, utils

In [ ]:
model_name = 'resnet50'
use_bias = False
importance_score = 'act'
batch_size = 128
eps=1e-6

num_shots = 1

seed = 42
curvature = 'sse'

target_pruning_ratio = 0.1
data_path = 'PATH/TO/DATA/FOLDER'

In [5]:
model = utils.load_model(model_name).cuda()
utils.assign_names(model)

_, test_data = utils.load_data(data_path, 256)

In [ ]:
train_data, _ = utils.load_data(data_path, 128, n_train_samples=1024, stratified=True, seed=13)
gram_matrices = gram_matrices.compute_gram_matrices(model, train_data, device='cuda')
utils.set_gram_matrices(model, gram_matrices)

100%|██████████| 8/8 [00:46<00:00,  5.84s/it]


In [ ]:
example_input = torch.randn(1, 3, 224, 224).cuda()
utils.set_num_patches(model, example_input)

ignored_layers = [model.fc]

DG = tp.DependencyGraph().build_dependency(
    model,
    example_inputs=example_input,
    ignored_layers=[],
    ignored_params=[],
)

local_scores, calibrated_scores = importances.compute_importance_scores(
    DG, ignored_layers, importance_score, importances.first_layer, use_bias, eps=1e-5, normalize=importance_score == 'act'
)

unit_orders_per_group = [
    torch.argsort(scores, descending=True) for scores in local_scores
]

In [ ]:
for idx in [6, 16, 19]:
    for ratio in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        model = utils.load_model(model_name).cuda()
        utils.assign_names(model)

        utils.set_gram_matrices(model, gram_matrices)

        DG = tp.DependencyGraph().build_dependency(
            model,
            example_inputs=example_input,
            ignored_layers=[],
            ignored_params=[],
        )
        ignored_layers = [model.fc]

        groups = list(pruning.pruning_groups(DG, ignored_layers))

        group, layers = groups[idx]
        
        n_units = layers[0].weight.shape[1]
        n_pruned = math.ceil(n_units * ratio)
        order = unit_orders_per_group[idx]
        units_pruned = order[n_units - n_pruned:]

        pruning.prune_group(group, layers, units_pruned, order, use_bias=False, reconstruct=True, eps=eps)

        flop, pc = tp.utils.count_ops_and_params(model, example_input)
        loss, acc = utils.measure_perf(model, nn.CrossEntropyLoss(), test_data, "cuda")

        print(idx, ratio, flop, pc, loss, acc)


100%|██████████| 196/196 [01:39<00:00,  1.98it/s]


6 0.3 4081596136.0 25544192 0.9692950961685181 0.75864


100%|██████████| 196/196 [02:28<00:00,  1.32it/s]


6 0.4 4069497448.0 25540340 0.9765286387634278 0.7571


100%|██████████| 196/196 [01:46<00:00,  1.84it/s]


6 0.5 4057398760.0 25536488 0.9841154456329345 0.75434


... (32 outputs left out)

100%|██████████| 196/196 [01:39<00:00,  1.96it/s]


19 0.8 3988085516.0 24874382 1.5362947492980956 0.63512


100%|██████████| 196/196 [02:00<00:00,  1.62it/s]

19 0.9 3971110740.0 24787802 2.0982413690948487 0.52874
